# 09 - Statelessness, Caching and Rate Limiting

## 1. Statelessness

**Definition:** The server **doesn't remember** anything between requests. Every request carries everything needed, like the JWT.

**Why it matters:** Any server instance can handle any request, so you can add more servers (horizontal scaling) behind a load balancer.

**Real-world example:** A user's first request hits server A, the next hits server B. Both work because the JWT travels with each request, not stored in server A's memory.

## 2. Caching

**Definition:** Storing a copy of frequently read data in fast memory so you don't hit the database every time.

**How it works (cache-aside):**
1. Check the cache (e.g. Redis)
2. **Hit** → return it immediately
3. **Miss** → read from the database, save in cache with a **TTL** (time to live), return it
4. When data changes, **delete (invalidate)** the cache entry

**Good to cache:** product lists, categories, config. **Avoid caching:** stock counts, payments, anything that must be exact right now.

**Real-world example:** A product page cached in Redis for 5 minutes. Thousands of users read it, but the database is hit only once every 5 minutes.

**Interview line:** "REST is stateless, so any server can handle any request. I use Redis with a TTL for read-heavy data and invalidate the cache when the data changes."

In [ ]:
# ==================================================
# 0. STATELESS vs STATEFUL - two servers behind a load balancer
# ==================================================
# STATEFUL (BAD): login stored in one server's memory
server_a_sessions = {"session_1": 5}
server_b_sessions = {}

def stateful_call(server_sessions, session_id):
    return 200 if session_id in server_sessions else 401

print("=" * 55)
print("[STATEFUL]  login on A, next request goes to A ->", stateful_call(server_a_sessions, "session_1"))
print("[STATEFUL]  next request goes to B            ->", stateful_call(server_b_sessions, "session_1"), "(user logged out!)")

# STATELESS (GOOD): the token itself carries the user
def stateless_call(server_name, token):
    user_id = token.get("user_id")          # in real life: verify JWT signature
    return 200 if user_id else 401

token = {"user_id": 5}
print("[STATELESS] request goes to A ->", stateless_call("A", token))
print("[STATELESS] request goes to B ->", stateless_call("B", token))
print("=" * 55)

In [ ]:
# ==================================================
# 1. CACHE-ASIDE WITH TTL (Redis-like, in memory)
# ==================================================
import time

database = {"product:5": {"id": 5, "name": "Laptop", "price": 55000}}
cache = {}          # key -> (value, expiry)
TTL = 2             # seconds (e.g. 300 in real life)
db_hits = 0

def get_product(key):
    global db_hits
    if key in cache and cache[key][1] > time.time():
        print(f"[HIT]   {key} from cache")
        return cache[key][0]
    db_hits += 1
    value = database[key]
    cache[key] = (value, time.time() + TTL)
    print(f"[MISS]  {key} from database -> cached for {TTL}s")
    return value

def update_price(key, price):
    database[key]["price"] = price
    cache.pop(key, None)            # invalidate
    print(f"[WRITE] {key} price={price} -> cache invalidated")

print("=" * 55)
get_product("product:5")
get_product("product:5")
get_product("product:5")
update_price("product:5", 52000)
print("        ->", get_product("product:5"))
time.sleep(TTL + 1)
get_product("product:5")            # TTL expired
print("=" * 55)
print(f"[RESULT] database hits: {db_hits} out of 5 reads")

## 3. Rate Limiting

**Definition:** Limiting how many requests a client can make in a time window, to protect the server from abuse, bots and overload.

**How it works (fixed window):**
- Track requests per user or IP, e.g. 5 per minute
- Under the limit → allow
- Over the limit → return **429 Too Many Requests** with a **Retry-After** header

**Other algorithms (names to know):** sliding window, token bucket (allows short bursts).

**Where:** API gateway, Nginx, or app middleware, usually with counters in Redis so all servers share them.

**Real-world example:** A login endpoint allows 5 attempts per minute per IP, which stops password-guessing bots. OpenAI and Azure OpenAI APIs return 429 when you exceed your quota.

**Interview line:** "I rate limit per user or IP, return 429 with Retry-After when exceeded, and keep counters in Redis so the limit works across all servers."

In [ ]:
# ==================================================
# 2. FIXED-WINDOW RATE LIMITER
# ==================================================
import time

LIMIT = 5         # requests
WINDOW = 60       # seconds
counters = {}     # ip -> (window_start, count)

def rate_limit(ip):
    now = time.time()
    start, count = counters.get(ip, (now, 0))
    if now - start >= WINDOW:          # new window
        start, count = now, 0
    if count >= LIMIT:
        retry_after = int(WINDOW - (now - start))
        return 429, f"Too Many Requests, Retry-After: {retry_after}s"
    counters[ip] = (start, count + 1)
    return 200, f"OK ({count + 1}/{LIMIT})"

print("=" * 55)
for i in range(1, 8):
    status, msg = rate_limit("10.0.0.1")
    print(f"[REQ {i}] IP 10.0.0.1 -> [{status}] {msg}")
print(f"[REQ 8] IP 10.0.0.2 -> {rate_limit('10.0.0.2')}  (different IP, own limit)")
print("=" * 55)

## Key Takeaways

- Stateless: no session in server memory; any server can handle any request
- Cache-aside: check cache → miss → read DB → store with TTL
- Invalidate the cache when data changes; don't cache exact-now data like stock
- Rate limit per user/IP; return 429 with Retry-After
- Keep cache and rate-limit counters in Redis so all servers share them